# <font color="DarkBlue"><center><bf>GPT vs human legal texts annotations: A comparative study with privacy policies</bf></font>

## <font color="DarkGreen"><center>Escuela Politécnica Nacional - Universidad Politécnica de Madrid</font>

**Authors:** David Cevallos. José Estrada-Jiménez, Danny S. Guamán, David Rodriguez, Jose M. Del Alamo

**Date:** 2024-11-15

# OPP-115 data

In [ ]:
# Clone and unzip OPP-115 corpus
!git clone https://github.com/dcevallossalas/opp-115.git
!unzip -u /content/opp-115/opp-115.zip -d /home/opp-115

Se han truncado las últimas 5000 líneas del flujo de salida.
  inflating: /home/opp-115/original_policies/1683_www.dailynews.com_files/sync  
  inflating: /home/opp-115/original_policies/1683_www.dailynews.com_files/t.gif  
  inflating: /home/opp-115/original_policies/1683_www.dailynews.com_files/tag  
  inflating: /home/opp-115/original_policies/1683_www.dailynews.com_files/tag(1)  
  inflating: /home/opp-115/original_policies/1683_www.dailynews.com_files/tag.aspx  
  inflating: /home/opp-115/original_policies/1683_www.dailynews.com_files/tag.js  
  inflating: /home/opp-115/original_policies/1683_www.dailynews.com_files/tout-functions.js  
  inflating: /home/opp-115/original_policies/1683_www.dailynews.com_files/tracker.js  
  inflating: /home/opp-115/original_policies/1683_www.dailynews.com_files/tracker_0.0_STABLE.js  
  inflating: /home/opp-115/original_policies/1683_www.dailynews.com_files/trk.js  
  inflating: /home/opp-115/original_policies/1683_www.dailynews.com_files/ttj  
  i

In [ ]:
# Define list of files for analysis
import os
files = os.listdir('/home/opp-115/annotations')
files.sort()

In [ ]:
# Define list of privacy policies and their indexes of starting and ending
import csv
import json
import numpy as np
import statistics

count = -1
ref = -1
policies = list()
for f in files:
  with open("/home/opp-115/annotations/" + f, "r") as handle:
    spamreader = csv.reader(handle, delimiter=',')
    max = 0

    for row in spamreader:
      if int(row[4]) > max:
        max = int(row[4])

    count = count + 1
    pol = [count, f, ref + 1, ref + max + 1]
    policies.append(pol)
    ref = ref + max + 1

kappa_threshold = 0.61
num_categories = 0
kappas = [0.76,0.76,0.61,0.74,0.55,0.67,0.73,0.87,0.91,0.49]

for kappa in kappas:
  if kappa > kappa_threshold:
    num_categories = num_categories + 1

# Ground truth definition

In [ ]:
# Define ground truth
gt = list()

for n in range(0,10):
  list_category = list()

  for m in range (0,ref+1):
    list_category.append(0)
  gt.append(list_category)

def set_label(policy, handle):
  spamreader = csv.reader(handle, delimiter=',')
  for row in spamreader:

    if row[0].startswith("C"):
      ind = 0
      if (row[5] == "First Party Collection/Use"):
        ind = 0
      elif (row[5] == "Third Party Sharing/Collection"):
        ind = 1
      elif (row[5] == "User Choice/Control"):
        ind = 2
      elif (row[5] == "User Access, Edit and Deletion"):
        ind = 3
      elif (row[5] == "Data Retention"):
        ind = 4
      elif (row[5] == "Data Security"):
        ind = 5
      elif (row[5] == "Policy Change"):
        ind = 6
      elif (row[5] == "International and Specific Audiences"):
        ind = 7
      elif (row[5] == "Do Not Track"):
        ind = 8
      elif (row[5] == "Other"):
        ind = 9
      else:
        print("** No considered: " + row[5])

      gt[ind][policy[2]+int(row[4])] = 1

def annotated(start,segment):
  for j in range(0,num_categories):
    if gt[j][start+segment] == 1:
      return True
  return False

def analyze_consolidation(folder,policy):
  with open("/home/opp-115/consolidation/" + folder + "/" + policy[1], "r") as handle:
    set_label(policy, handle)

for policy in policies:
  analyze_consolidation("threshold-0.75-overlap-similarity", policy)

# Humans annotations definition

In [ ]:
# Determine coders
ids_coders=list()

for policy in policies:
  with open("/home/opp-115/annotations/" + policy[1], "r") as handle:
    spamreader = csv.reader(handle, delimiter=',')

    for row in spamreader:
      if int(row[2]) not in ids_coders:
        ids_coders.append(int(row[2]))

ids_coders.sort()
print(ids_coders)

# Set default values for coders' annotations
coders = list()

for id_coder in ids_coders:
  coder = list()

  for n in range(0,10):
    category = list()

    for m in range (0,ref+1):
      category.append(-1)
    coder.append(category)

  coders.append(coder)

# Set default coders' annotations for policies in which they intervene
for policy in policies:
  with open("/home/opp-115/annotations/" + policy[1], "r") as handle:
    spamreader = csv.reader(handle, delimiter=',')

    for row in spamreader:
      idx = ids_coders.index(int(row[2]))

      for cat in range(0,10):
        for order in range(policy[2], policy[3]+1):
          coders[idx][cat][order]=0

# Set coder's annotations
for policy in policies:
  with open("/home/opp-115/annotations/" + policy[1], "r") as handle:
    spamreader = csv.reader(handle, delimiter=',')

    for row in spamreader:
      idx = ids_coders.index(int(row[2]))
      ind = 0

      if (row[5] == "First Party Collection/Use"):
        ind = 0
      elif (row[5] == "Third Party Sharing/Collection"):
        ind = 1
      elif (row[5] == "User Choice/Control"):
        ind = 2
      elif (row[5] == "User Access, Edit and Deletion"):
        ind = 3
      elif (row[5] == "Data Retention"):
        ind = 4
      elif (row[5] == "Data Security"):
        ind = 5
      elif (row[5] == "Policy Change"):
        ind = 6
      elif (row[5] == "International and Specific Audiences"):
        ind = 7
      elif (row[5] == "Do Not Track"):
        ind = 8
      elif (row[5] == "Other"):
        ind = 9
      else:
        print("** No considered: " + row[5])

      coders[idx][ind][policy[2]+int(row[4])]=1


[82, 84, 88, 95, 103, 116, 117, 118, 121, 123]


# Humans segment levels annotation performance

In [ ]:
# F1-score calculation for segment-level annotations
from sklearn.metrics import f1_score

def calculate_f1score(gt,coders,gpt=None):
  cod = -1
  total = list()

  for coder in coders:
    cod = cod+1
    print("-------------------------------------------------------------------")
    if gpt is None:
      print("Coder " + str(cod+1) + " (ID " + str(ids_coders[cod]) + ")")
    else:
      print("Gpt metrics for privacy policies coded by coder " + str(cod+1) + " (ID " + str(ids_coders[cod]) + ")")
    print("-------------------------------------------------------------------")
    cat = -1
    f1s = list()

    for category in coder:
      cat = cat+1
      ytrue = list()
      ypred = list()
      dat = -1

      for label in category:
        dat = dat+1

        if label >= 0:
          ytrue.append(gt[cat][dat])
          if gpt is None:
            ypred.append(coders[cod][cat][dat])
          else:
            ypred.append(gpt[cat][dat])

      f1 = f1_score(ytrue,ypred)
      f1s.append(f1)

      if cat == 0 and kappas[0] > kappa_threshold:
        print("F1-score category First Party Collection/Use: " + str(f1))
      elif cat == 1 and kappas[1] > kappa_threshold:
        print("F1-score category Third Party Sharing/Collection: " + str(f1))
      elif cat == 2 and kappas[2] > kappa_threshold:
        print("F1-score category User Choice/Control: " + str(f1))
      elif cat == 3 and kappas[3] > kappa_threshold:
        print("F1-score category User Access, Edit and Deletion: " + str(f1))
      elif cat == 4 and kappas[4] > kappa_threshold:
        print("F1-score category Data Retention: " + str(f1))
      if cat == 5 and kappas[5] > kappa_threshold:
        print("F1-score category Data Security: " + str(f1))
      elif cat == 6 and kappas[6] > kappa_threshold:
        print("F1-score category Policy Change: " + str(f1))
      elif cat == 7 and kappas[7] > kappa_threshold:
        print("F1-score category International and Specific Audiences: " + str(f1))
      elif cat == 8 and kappas[8] > kappa_threshold:
        print("F1-score category Do Not Track: " + str(f1))
      elif cat == 9 and kappas[9] > kappa_threshold:
        print("F1-score category Other: " + str(f1))

    # Filtering categories by Fleiss' Kappa values
    f1s_kappa = list()
    k_c = -1
    for kappa in kappas:
      k_c = k_c + 1
      if kappa > kappa_threshold:
        f1s_kappa.append(f1s[k_c])
    total.append(f1s_kappa)

    mean = statistics.mean(f1s_kappa)
    sd = statistics.pstdev(f1s_kappa)
    median = statistics.median(f1s_kappa)
    print("Mean f1-score: " + str(mean))
    print("Standard deviation f1-score: " + str(sd))
    print("Median f1-score: " + str(median))
  return total

def save_csv(path,values):
  with open(path,"w+") as handle:
    handle.write("coder1,coder2,coder3,coder4,coder5,coder6,coder7,coder8,coder9,coder10\n")
    for cat in range(0,num_categories):
      row = str(values[0][cat])+","+str(values[1][cat])+","+str(values[2][cat])+","+str(values[3][cat])+","+ \
      str(values[4][cat])+","+str(values[5][cat])+","+str(values[6][cat])+","+str(values[7][cat])+","+ \
      str(values[8][cat])+","+str(values[9][cat])
      if cat < num_categories-1:
        handle.write(row+"\n")
      else:
        handle.write(row)

print("Humans segment level annotations")
print("---------------------------------------------------------------------")

f1_coders=calculate_f1score(gt,coders)
save_csv("/home/segments_coders.csv",f1_coders)

Humans segment level annotations
---------------------------------------------------------------------
-------------------------------------------------------------------
Coder 1 (ID 82)
-------------------------------------------------------------------
F1-score category First Party Collection/Use: 0.31063829787234043
F1-score category Third Party Sharing/Collection: 0.21354166666666666
F1-score category User Access, Edit and Deletion: 0.5507246376811594
F1-score category Data Security: 0.8118811881188119
F1-score category Policy Change: 0.7941176470588235
F1-score category International and Specific Audiences: 0.8977272727272727
F1-score category Do Not Track: 0.9
Mean f1-score: 0.639804387160725
Standard deviation f1-score: 0.26339389363674487
Median f1-score: 0.7941176470588235
-------------------------------------------------------------------
Coder 2 (ID 84)
-------------------------------------------------------------------
F1-score category First Party Collection/Use: 0.3491271

# Humans whole-text level annotations performance

In [ ]:
# F1-score calculation for whole privacy-level annotations
gt_total = list()
coders_total = list()

for id_category in range(0,10):
  category_values = list()
  for policy in policies:
    value = sum(gt[id_category][policy[2]:policy[3]+1])
    if value > 0:
      category_values.append(1)
    else:
      category_values.append(0)
  gt_total.append(category_values)

for id_coder in range(0,len(ids_coders)):
  coder_values = list()
  for id_category in range(0,10):
    category_values = list()
    for policy in policies:
      value = sum(coders[id_coder][id_category][policy[2]:policy[3]+1])
      if value > 0:
        category_values.append(1)
      elif value == 0:
        category_values.append(0)
      else:
        category_values.append(-1)
    coder_values.append(category_values)
  coders_total.append(coder_values)

print("Humans whole-text level annotations")
print("---------------------------------------------------------------------")

result = calculate_f1score(gt_total,coders_total)
save_csv("/home/total_coders.csv",result)

Humans whole-text level annotations
---------------------------------------------------------------------
-------------------------------------------------------------------
Coder 1 (ID 82)
-------------------------------------------------------------------
F1-score category First Party Collection/Use: 0.90625
F1-score category Third Party Sharing/Collection: 0.7272727272727273
F1-score category User Access, Edit and Deletion: 0.7142857142857143
F1-score category Data Security: 0.9818181818181818
F1-score category Policy Change: 0.8571428571428571
F1-score category International and Specific Audiences: 0.9811320754716981
F1-score category Do Not Track: 0.9
Mean f1-score: 0.8668430794273112
Standard deviation f1-score: 0.1012643377709898
Median f1-score: 0.9
-------------------------------------------------------------------
Coder 2 (ID 84)
-------------------------------------------------------------------
F1-score category First Party Collection/Use: 0.9206349206349206
F1-score catego

# GPT segments level annotations performance

In [ ]:
# Gpt annotations analysis
def define_gpt_annotations(thresold):
  gpt_annotations = list()

  for n in range(0,10):
    gpt_list_category = list()

    for m in range (0,ref+1):
      gpt_list_category.append(0)
    gpt_annotations.append(gpt_list_category)

  for i in range(0,71):
    start = policies[i][2]


    with open("/home/opp-115/gpt_annotations/segments_level/segments_gpt_" + str(i) + ".json", "r") as handle:
      result = json.load(handle)
      for segment in result["segments"]:
        flag = False
        for j in range(0,10):
          if gt[j][start+segment["segment"]] == 1:
            flag = True
        if flag:
          for category in segment["categories"]:
            if str(category["category"]).strip().lower() != "none" and annotated(start,segment["segment"]) and np.exp(category["log_prob"]) > threshold:
              gpt_annotations[category["category"]][start+segment["segment"]] = 1

  return gpt_annotations

thresholds = [0,0.8,0.85,0.9,0.95]

print("GPT segments level annotations")
print("---------------------------------------------------------------------")

for threshold in thresholds:
  lab = "Threshold " + str(threshold)
  print(lab)
  f1_gpt=calculate_f1score(gt,coders,define_gpt_annotations(threshold))
  save_csv("/home/segments_th"+str(threshold).replace(".","")+".csv",f1_gpt)
  print("*******************************************************************")
  print("*******************************************************************")

GPT segments level annotations
---------------------------------------------------------------------
Threshold 0
-------------------------------------------------------------------
Gpt metrics for privacy policies coded by coder 1 (ID 82)
-------------------------------------------------------------------
F1-score category First Party Collection/Use: 0.7801418439716312
F1-score category Third Party Sharing/Collection: 0.5806451612903226
F1-score category User Access, Edit and Deletion: 0.8292682926829268
F1-score category Data Security: 0.825
F1-score category Policy Change: 0.8076923076923077
F1-score category International and Specific Audiences: 0.023255813953488372
F1-score category Do Not Track: 0.0
Mean f1-score: 0.5494290599415252
Standard deviation f1-score: 0.34926098606812356
Median f1-score: 0.7801418439716312
-------------------------------------------------------------------
Gpt metrics for privacy policies coded by coder 2 (ID 84)
-----------------------------------------

# GPT whole-text level annotations performance

In [ ]:
# Gpt whole-text level annotations analysis
def define_gpt_total_annotations(threshold):
  gpt_annotations = list()

  for n in range(0,10):
    gpt_list_category = list()

    for m in range (0,115):
      gpt_list_category.append(0)
    gpt_annotations.append(gpt_list_category)

  for i in range(0,12):
    start = i*10
    end = start+10

    if end > 115:
      end = 115

    with open("/home/opp-115/gpt_annotations/whole_text_level/total_gpt_" + str(start) + "_" + str(end-1) + ".json","r") as handle:
      results = json.load(handle)
      for result in results:
        for category in result["categories"]:
          if str(category["category"]).strip().lower() != "none" and np.exp(category["log_prob"]) > threshold:
              gpt_annotations[category["category"]][result["policy"]] = 1

  return gpt_annotations

thresholds = [0,0.8,0.85,0.9,0.95]

print("GPT whole-text level annotations")
print("---------------------------------------------------------------------")

for threshold in thresholds:
  print("Threshold " + str(threshold))
  f1_gpt_total=calculate_f1score(gt_total,coders_total,define_gpt_total_annotations(threshold))
  save_csv("/home/total_th"+str(threshold).replace(".","")+".csv",f1_gpt_total)
  print("*******************************************************************")
  print("*******************************************************************")

GPT whole-text level annotations
---------------------------------------------------------------------
Threshold 0
-------------------------------------------------------------------
Gpt metrics for privacy policies coded by coder 1 (ID 82)
-------------------------------------------------------------------
F1-score category First Party Collection/Use: 0.9206349206349206
F1-score category Third Party Sharing/Collection: 0.7272727272727273
F1-score category User Access, Edit and Deletion: 0.6666666666666666
F1-score category Data Security: 0.9310344827586207
F1-score category Policy Change: 0.8235294117647058
F1-score category International and Specific Audiences: 0.9454545454545454
F1-score category Do Not Track: 0.782608695652174
Mean f1-score: 0.8281716357434801
Standard deviation f1-score: 0.10088712347370819
Median f1-score: 0.8235294117647058
-------------------------------------------------------------------
Gpt metrics for privacy policies coded by coder 2 (ID 84)
--------------